In [8]:
#create one data set

import pandas as pd
import numpy as np

START = "2014-05-31"
END = "2026-05-31"

PILLAR_FILE = "../Macro Scores/Macro Pillars Attempt 2.xlsx"
FACTOR_FILE = "monthly_factor_returns.csv"
OUTPUT_FILE = "theme_pillar_merged.csv"

PILLAR_COLS = ["Liquidity_Adjusted_Net_Score", "Inflation_Net_Composite_Score", "Growth_Adjusted_Net_Score", "External_Stress_Net_Score", "Rates_Final_Net_Score"]
FACTOR_COLS = [
    "earnings", "management", "momentum", "profitability",
    "quality", "sentiment main", "Reversal", "value",
]

# ===========================================================================
# 1. Load + minimal cleaning
# ===========================================================================
pillar = pd.read_excel(PILLAR_FILE, sheet_name="Sheet1")
pillar["Date"] = pd.to_datetime(pillar["Date"]).dt.normalize() + pd.offsets.MonthEnd(0)
pillar = pillar[["Date"] + PILLAR_COLS]

factors = pd.read_csv(FACTOR_FILE)
factors["datetime"] = pd.to_datetime(factors["datetime"]).dt.normalize() + pd.offsets.MonthEnd(0)
factors = factors.rename(columns={"datetime": "Date"})

# ===========================================================================
# 2. Sort chronologically -- before any restriction or calculation
# ===========================================================================
pillar = pillar.sort_values("Date").reset_index(drop=True)
factors = factors.sort_values("Date").reset_index(drop=True)

# ===========================================================================
# 3. Restrict to the target period -- before any calculation
# ===========================================================================
mask_p = (pillar["Date"] >= START) & (pillar["Date"] <= END)
mask_f = (factors["Date"] >= START) & (factors["Date"] <= END)
pillar = pillar.loc[mask_p].reset_index(drop=True)
factors = factors.loc[mask_f].reset_index(drop=True)

# ===========================================================================
# 4. Sanity checks on the restricted period
# ===========================================================================
full_index = pd.date_range(START, END, freq="ME")
missing_in_pillars = set(full_index) - set(pillar["Date"])
missing_in_factors = set(full_index) - set(factors["Date"])
if missing_in_pillars:
    print(f"WARNING: {len(missing_in_pillars)} month(s) missing from Macro Pillars Attempt 2.xlsx: "
          f"{sorted(d.date() for d in missing_in_pillars)}")
if missing_in_factors:
    print(f"WARNING: {len(missing_in_factors)} month(s) missing from monthly_factor_returns.csv: "
          f"{sorted(d.date() for d in missing_in_factors)}")

dup_p = pillar["Date"].duplicated().sum()
dup_f = factors["Date"].duplicated().sum()
if dup_p:
    print(f"WARNING: {dup_p} duplicate date(s) in Regime_Output.xlsx")
if dup_f:
    print(f"WARNING: {dup_f} duplicate date(s) in monthly_factor_returns.csv")

# ===========================================================================
# 5. Merge (both inputs already sorted + restricted to the target period)
# ===========================================================================
merged = pd.merge(pillar, factors, on="Date", how="inner").sort_values("Date").reset_index(drop=True)

# ===========================================================================
# 6. Lag + octant-validity mask
# ===========================================================================
pillar = pillar.shift(1)

# ===========================================================================
# 7. Calculations / transformations
# ===========================================================================
ACROSS_RELATIVE_BASE_COLS = [col for col in FACTOR_COLS if col != "Reversal" and col != "sentiment main"]
SELF_RELATIVE_BASE_COLS = [col for col in FACTOR_COLS]

# Cross-sectional mean -- ROW-WISE (axis=1) across themes in the SAME month.
# Not a historical/time-series baseline, so it isn't affected by the bug
# above and needs no period-gating.
merged["factor_mean_across"] = merged[ACROSS_RELATIVE_BASE_COLS].mean(axis=1)

ACROSS_RELATIVE_COLS = []
SELF_RELATIVE_COLS = []

for col in ACROSS_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_across_relative"
    merged[rel_col_name] = merged[col] - merged["factor_mean_across"]
    ACROSS_RELATIVE_COLS.append(rel_col_name)

theme_means = {}
for col in SELF_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_theme_relative"

    # Scalar historical mean for this theme, restricted to the 6-octant
    # universe (was: factors[col].mean() over the full, unmerged,
    # unrestricted 2008-2026 file -- the bug).
    theme_mean = merged.loc[mask_p, col].mean()
    theme_means[col] = theme_mean

    merged[rel_col_name] = merged[col] - theme_mean
    SELF_RELATIVE_COLS.append(rel_col_name)

print("\nSelf-relative baselines (theme_mean), computed on the 6-octant universe only:")
for col, m in theme_means.items():
    print(f"  {col:16s}: {m:+.6f}")

# ===========================================================================
# 8. Verification
# ===========================================================================
# N-weighted average of each theme's self-relative return, across just the
# 6 valid octants, should now be ~0 by construction -- not exactly 0 to
# machine precision if a theme has scattered NaNs shifting denominators
# slightly, but nowhere near the systematic all-one-side result the old
# baseline produced.
print("\nVerification -- N-weighted avg Self-Relative return across the 6 octants "
      "(should be ~0 for every theme):")
for col in SELF_RELATIVE_BASE_COLS:
    rel_col_name = f"{col}_theme_relative"
    sub = merged.loc[mask_p, [rel_col_name]].dropna()
    weighted_avg = sub[rel_col_name].mean()  # equal-weighted across valid-octant months = N-weighted across octants
    print(f"  {col:16s}: {weighted_avg:+.8f}")

# ===========================================================================
# 9. Save
# ===========================================================================
merged = merged.drop(columns=["factor_mean_across"])
merged = merged.rename(columns={"Date": "Month_End"})
merged = merged[["Month_End"] + PILLAR_COLS + FACTOR_COLS + ACROSS_RELATIVE_COLS + SELF_RELATIVE_COLS]

merged.to_csv(OUTPUT_FILE, index=False)

print(f"\nMerged shape: {merged.shape}")
print(f"Date range in output: {merged['Month_End'].min().date()} to {merged['Month_End'].max().date()}")
print(f"Saved to: {OUTPUT_FILE}")


Self-relative baselines (theme_mean), computed on the 6-octant universe only:
  earnings        : +0.002530
  management      : +0.001824
  momentum        : +0.001398
  profitability   : -0.000553
  quality         : +0.000879
  sentiment main  : +0.006033
  Reversal        : +0.009625
  value           : +0.001866

Verification -- N-weighted avg Self-Relative return across the 6 octants (should be ~0 for every theme):
  earnings        : +0.00000000
  management      : -0.00000000
  momentum        : +0.00000000
  profitability   : +0.00000000
  quality         : +0.00000000
  sentiment main  : +0.00000000
  Reversal        : -0.00000000
  value           : -0.00000000

Merged shape: (145, 28)
Date range in output: 2014-05-31 to 2026-05-31
Saved to: theme_pillar_merged.csv
